### 把美联储经济新闻摘要成高中生能懂的话

## 练习目标（理念）

用户提供美联储相关 URL，用 **OpenAI Chat Completions** 把页面内容改写成高中生能懂的通俗摘要。

- **抓取**：`scraper.fetch_website_contents(url)`
- **角色**：耐心的经济学老师（system prompt）
- **输出**：要点子弹 + 1–3 段通俗解释（Markdown）

## 和本课 Day 1 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 环境变量 | `load_dotenv` + `OPENAI_API_KEY` |
| 网页抓取 | `fetch_website_contents` |
| system / user | 老师角色 +「页面正文 + 摘要任务」 |
| Chat Completions | `gpt-4.1-mini` |

## 怎么跑

1. 确保能 `import scraper`（与本 notebook 同目录或已在路径中）
2. `.env` 配好 `OPENAI_API_KEY`
3. 从上到下运行；可把 URL 换成其他 FRED / 美联储页面做对比


In [ ]:
# ========== 导入：环境、抓取工具、展示、OpenAI 客户端 ==========

# 导入标准库 os：读环境变量 OPENAI_API_KEY
import os
# 导入标准库 io：本格导入了（保持原样）；后面单元格未直接使用
import io
# 从 dotenv 导入 load_dotenv：把 .env 密钥读进环境变量
from dotenv import load_dotenv
# 从 scraper 导入 fetch_website_contents：抓取网页正文的课程工具函数
from scraper import fetch_website_contents
# 从 IPython.display 导入 Markdown / display：漂亮渲染模型回答
from IPython.display import Markdown, display
# 从 openai 导入 OpenAI 客户端类
from openai import OpenAI


In [ ]:
# ========== 创建 OpenAI 客户端（默认读环境变量 OPENAI_API_KEY）==========

# 无参构造客户端；密钥通常来自环境变量
openai= OpenAI()


In [ ]:
# ========== 加载并检查 OPENAI_API_KEY ==========

# override=True：用 .env 覆盖进程里已有同名变量
load_dotenv(override = True)
# 取出密钥字符串
api_key = os.getenv('OPENAI_API_KEY')

# 三段校验；失败时的英文提示保持原样（便于对照 troubleshooting）
if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


In [ ]:
# ========== 抓取示例：圣路易斯联储 FRED 的 M2 货币供应量页面 ==========

# fetch_website_contents：返回页面文本；URL 决定抓取目标，保持原样
fednuu = fetch_website_contents("https://fred.stlouisfed.org/series/M2SL")

# 先打印原始抓取结果，确认 scraper 工作正常
print(fednuu)


In [ ]:
# ========== system prompt：高中生能懂的美联储新闻讲解老师 ==========

# 发给模型的角色与格式要求——英文指令不翻译（改译会改变回答风格）
system_prompt = """
You are a patient systematic economics teacher.
You read official Federal Reserve url, and explain the key economic news in simple language for a high school student.

When you summarize:
- Start with 3–5 short bullet points of the most important news.
- Then add 1–3 short paragraphs that explain what it means for the economy
  and for ordinary people (e.g. jobs, prices, borrowing costs).
- Avoid jargon where possible; briefly explain terms like inflation,
  interest rates, unemployment, and GDP if you must use them.
- Do not quote long passages; paraphrase in clear, plain English.

Respond in markdown (no code blocks).
"""


In [ ]:
# ========== user 提示前缀：说明任务后接「页面正文」==========

# 前缀本身是发给模型的英文任务说明，不翻译；后面会拼上 website 文本
user_prompt_prefix = """
You will be given the url of a page from the official federal reserve website
Your task is to:
- Summarize the most important economic news in a way a high school
  student can understand
- Focus on Money supply, interest rates, inflation, jobs, growth, and any future guidance

First give 3–5 bullet points of the key takeaways,
then a short explanation in 1–3 paragraphs.

Here is the page text:
"""


In [ ]:
# ========== messages_for：组装 Chat Completions 所需的 messages ==========

# 辅助函数：把 system +（前缀 + 网页正文）拼成 API 期望的列表
def messages_for(website):
    return [ 
        # system：经济学老师角色与输出格式
        {"role" :"system" , "content": system_prompt},
        # user：任务前缀 + 抓取到的页面文本（参数名 website，实际是字符串正文）
        {"role": "user", "content": user_prompt_prefix + website }
        
        ]


In [ ]:
# ========== 预览：看拼好的 messages 结构 ==========

# 对刚才抓到的 M2 页面正文组装 messages
messages_for(fednuu)


In [ ]:
# ========== summarize：URL → 抓取 → 调 gpt-4.1-mini → 返回摘要 ==========

def summarize(url):
    # 抓取目标页正文
    website = fetch_website_contents(url)
    # 调用云端模型；model id 保持原样
    response = openai.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = messages_for(website)
    )
    # 取出助手回复正文
    return response.choices[0].message.content


In [ ]:
# ========== 端到端：摘要 FRED M2 页面 ==========

summarize("https://fred.stlouisfed.org/series/M2SL")


In [ ]:
# ========== display_summary：用 Markdown 漂亮展示摘要 ==========

# 先 summarize，再 display(Markdown(...))
def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))


In [ ]:
# ========== 货币供应量示例：Markdown 渲染展示 ==========

# Money Supply Example：同一 FRED M2 URL，看渲染后的通俗讲解
display_summary("https://fred.stlouisfed.org/series/M2SL")
